# inference using Emotion2Vec

In [168]:
cd Prosody2Vec/

[Errno 2] No such file or directory: 'Prosody2Vec/'
/home/dcor/niskhizov/Prosody2Vec


In [191]:
import torch
from torch import nn
import torch 
import glob
from IPython.display import clear_output, display, Audio
import copy
import torch

from torch.optim import Adam
from torch.nn.functional import l1_loss

from tqdm import tqdm_notebook,tqdm
import plotly.express as px
import torchaudio
from funasr import AutoModel

sr = 16000


In [171]:


class FusionDecoderV1(nn.Module):
    def __init__(self, hidden_dim, acoustic):
        super(FusionDecoderV1, self).__init__()
        
        self.ff1 = nn.Linear(1024, 512).cuda()
        self.ff2 = nn.Linear(1024, 512).cuda()

        self.base_model = copy.deepcopy(acoustic)

    def forward(self, units, emo_vecs,  logmels):
        # units: (batch_size, time, hidden_dim)
        # emo_vecs: (batch_size, emo_vec_size)
        # spk_vecs: (batch_size, spk_vec_size)
        # logmels: (batch_size, time, n_mels)
        
        # batch_size, time, _ = units.shape
        
        o = self.base_model.encoder(units.cuda())

        o2 = self.ff1(emo_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)

        o3 = self.ff2(torch.cat([o, o2], dim=-1))

        d = self.base_model.decoder(o3, logmels)

        return d
    
    def generate(self, units, emo_vecs):
        # units: (batch_size, time, hidden_dim)
        # emo_vecs: (batch_size, emo_vec_size)
        # spk_vecs: (batch_size, spk_vec_size)
                
        o = self.base_model.encoder(units.cuda())

        # units_attn = self.attn(o, emo_vecs)
        o2 = self.ff1(emo_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        # units_attn = self.attn(o, emo_vecs)  # (batch_size, time, hidden_dim)

        o3 = self.ff2(torch.cat([o, o2], dim=-1))


        d = self.base_model.decoder.generate(o3)
        
        return d


In [455]:
acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_discrete", trust_repo=True).cuda()
decoderv1 = FusionDecoderV1(512, acoustic).cuda()

Using cache found in /home/dcor/niskhizov/cache/hub/bshall_acoustic-model_main


In [456]:
decoderv1.load_state_dict(torch.load("decoderV1.pth"))

/tmp/ipykernel_288879/427993933.py:1: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



<All keys matched successfully>

In [457]:
hifigan = torch.hub.load("bshall/hifigan:main", "hifigan_hubert_discrete", trust_repo=True).cuda()
hubert_discrete = torch.hub.load("bshall/hubert:main", "hubert_discrete", trust_repo=True).cuda()
model_id = "iic/emotion2vec_plus_large"

sed_model = AutoModel(
    model=model_id,
    hub="ms",  # "ms" or "modelscope" for China mainland users; "hf" or "huggingface" for other overseas users
)

Using cache found in /home/dcor/niskhizov/cache/hub/bshall_hifigan_main
/home/dcor/niskhizov/anaconda3/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning:

`torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.

Using cache found in /home/dcor/niskhizov/cache/hub/bshall_hubert_main


2025-03-02 21:09:12,625 - modelscope - WARNING - Using branch: master as version is unstable, use with caution


Detect model requirements, begin to install it: /home/dcor/niskhizov/.cache/modelscope/hub/models/iic/emotion2vec_plus_large/requirements.txt
install model requirements successfully
ckpt: /home/dcor/niskhizov/.cache/modelscope/hub/models/iic/emotion2vec_plus_large/model.pt


/home/dcor/niskhizov/anaconda3/lib/python3.12/site-packages/funasr/train_utils/load_pretrained_model.py:68: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



init param, map: modality_encoders.AUDIO.extra_tokens from d2v_model.modality_encoders.AUDIO.extra_tokens in ckpt
init param, map: modality_encoders.AUDIO.alibi_scale from d2v_model.modality_encoders.AUDIO.alibi_scale in ckpt
init param, map: modality_encoders.AUDIO.local_encoder.conv_layers.0.0.weight from d2v_model.modality_encoders.AUDIO.local_encoder.conv_layers.0.0.weight in ckpt
init param, map: modality_encoders.AUDIO.local_encoder.conv_layers.0.2.1.weight from d2v_model.modality_encoders.AUDIO.local_encoder.conv_layers.0.2.1.weight in ckpt
init param, map: modality_encoders.AUDIO.local_encoder.conv_layers.0.2.1.bias from d2v_model.modality_encoders.AUDIO.local_encoder.conv_layers.0.2.1.bias in ckpt
init param, map: modality_encoders.AUDIO.local_encoder.conv_layers.1.0.weight from d2v_model.modality_encoders.AUDIO.local_encoder.conv_layers.1.0.weight in ckpt
init param, map: modality_encoders.AUDIO.local_encoder.conv_layers.1.2.1.weight from d2v_model.modality_encoders.AUDIO.loc

In [562]:

from speechbrain.inference.speaker import EncoderClassifier

spk_ecapa_tdnn = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb").to('cuda').eval()
spk_ecapa_tdnn.device = 'cuda'
acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_discrete", trust_repo=True).cuda()


def extract_embedding(wav_path):
    wav, sr = torchaudio.load(wav_path)

    # take 3 seconds of audio

    with torch.inference_mode():
        # Extract speech units
        discrite_units = hubert_discrete.units(wav.unsqueeze(0).cuda())
        
        emo_vec = torch.tensor(sed_model.generate(wav, granularity="utterance", extract_embedding=True, disable_pbar =True)[0]['feats'])

        spk_vec = spk_ecapa_tdnn.encode_batch(wav.cuda())

    return discrite_units, emo_vec, wav, spk_vec[0]



INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
DEBUG:speechbrain.utils.parameter_transfer:Fetching files for pretraining (no collection directory set)
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["embedding_model"] = /home/dcor/niskhizov/cache/hf/hub/models--speechbrain--spkrec-ecapa-voxceleb/snapshots/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286/embedding_model.ckpt
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["mean_var_norm_emb"] = /home/dcor/n

In [565]:
sad = "Emotion Speech Dataset/0018/Sad/0018_001305.wav"
angry = 'Emotion Speech Dataset/0018/Angry/0018_000578.wav'
happy = 'Emotion Speech Dataset/0018/Happy/0018_000752.wav'
surprise = "Emotion Speech Dataset/0018/Surprise/0018_001674.wav"

angry_male = "/home/dcor/niskhizov/Prosody2Vec/Emotion Speech Dataset/0020/Angry/0020_000577.wav"

embed_sad = extract_embedding(sad)
embed_angry = extract_embedding(angry)
embed_happy = extract_embedding(happy)
embed_surprise = extract_embedding(surprise)
embed_angry_male = extract_embedding(angry_male)


In [485]:
decoderv1 = decoderv1.eval()


def generate_speech(units, emo_vec):
        
        with torch.no_grad():
            spec = decoderv1.generate(units.unsqueeze(0).cuda(), emo_vec.unsqueeze(0).cuda())
            target = hifigan(spec.transpose(1, 2)).cpu()[0][0]

        return target, spec

In [486]:
generated_wav_sad2angry, spec = generate_speech(embed_sad[0], embed_angry[1])
generated_wav_sad2happy, spec = generate_speech(embed_sad[0], embed_happy[1])
generated_wav_sad2surprise, spec = generate_speech(embed_sad[0], embed_surprise[1])

In [487]:
display(Audio(generated_wav_sad2angry, rate=sr))
display(Audio(generated_wav_sad2happy, rate=sr))
display(Audio(generated_wav_sad2surprise, rate=sr))
display(Audio(embed_sad[2], rate=sr))
display(Audio(embed_angry[2], rate=sr))
display(Audio(embed_happy[2], rate=sr))


# Inference using Emotion pretraining

In [488]:


class FusionDecoderV2(nn.Module):
    def __init__(self, hidden_dim, acoustic, spk_ecapa_tdnn):
        super(FusionDecoderV2, self).__init__()
        # self.attn = AttentionFusion(prosody_dim, hidden_dim)
        
        self.ff1 = nn.Linear(192, 512).cuda()
        self.ff2 = nn.Linear(1024, 512).cuda()

        self.base_model = copy.deepcopy(acoustic)

        self.ecapa = spk_ecapa_tdnn

    def forward(self, units, wav,  logmels):
        # units: (batch_size, time, hidden_dim)
        # emo_vecs: (batch_size, emo_vec_size)
        # spk_vecs: (batch_size, spk_vec_size)
        # logmels: (batch_size, time, n_mels)
        
        # batch_size, time, _ = units.shape
        
        # Apply attention
        o = self.base_model.encoder(units.cuda())

        emo_vecs = self.ecapa.encode_batch(wav.cuda()).squeeze(1)

        o2 = self.ff1(emo_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        # units_attn = self.attn(o, emo_vecs)  # (batch_size, time, hidden_dim)

        o3 = self.ff2(torch.cat([o, o2], dim=-1))

        d = self.base_model.decoder(o3, logmels)

        return d
    
    def generate(self, units, wav):
        # units: (batch_size, time, hidden_dim)
        # emo_vecs: (batch_size, emo_vec_size)
        # spk_vecs: (batch_size, spk_vec_size)
                
        # Apply attention
        o = self.base_model.encoder(units.cuda())

        emo_vecs = self.ecapa.encode_batch(wav.cuda()).squeeze(1)

        # units_attn = self.attn(o, emo_vecs)
        o2 = self.ff1(emo_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        # units_attn = self.attn(o, emo_vecs)  # (batch_size, time, hidden_dim)

        o3 = self.ff2(torch.cat([o, o2], dim=-1))


        d = self.base_model.decoder.generate(o3)
        
        return d


In [489]:
decoderv2 = FusionDecoderV2(512, acoustic, spk_ecapa_tdnn).cuda()

decoderv2.load_state_dict(torch.load("decoderV2.pth"))

/tmp/ipykernel_288879/1736823993.py:3: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



<All keys matched successfully>

In [490]:
def generate_speechv2(units, wav):
        
        with torch.no_grad():
            spec = decoderv2.generate(units.unsqueeze(0).cuda(), wav.cuda())
            target = hifigan(spec.transpose(1, 2)).cpu()[0][0]

        return target, spec

In [491]:


generated_wav_sad2angry, spec = generate_speechv2(embed_sad[0], embed_angry[2])
generated_wav_sad2happy, spec = generate_speechv2(embed_sad[0], embed_happy[2])
generated_wav_sad2surprise, spec = generate_speechv2(embed_sad[0], embed_surprise[2])

display(Audio(generated_wav_sad2angry, rate=sr))
display(Audio(generated_wav_sad2happy, rate=sr))
display(Audio(generated_wav_sad2surprise, rate=sr))
display(Audio(embed_sad[2], rate=sr))
display(Audio(embed_angry[2], rate=sr))



In [492]:
data_dir = '/home/dcor/niskhizov/Prosody2Vec/Emotion Speech Dataset/0018/'
# scan recursively for all .wav files in the data_dir
wav_files = glob.glob(data_dir + '/**/*.wav', recursive=True)


In [493]:
emo_vecs = []
for wav_path in tqdm(wav_files):
    with torch.no_grad():
        wav,sr = torchaudio.load(wav_path)
        wav = wav[:, :]
        out = decoderv2.ecapa.encode_batch(wav.cuda()).squeeze(1)
        emo_vecs.append([out.cpu().detach().numpy(), wav_path])

  0%|          | 0/1750 [00:00<?, ?it/s]

100%|██████████| 1750/1750 [00:40<00:00, 42.78it/s]


In [494]:
emo_vecs_np = np.stack([x[0] for x in emo_vecs])
# do tsne analysis on the embeddings to see if they cluster 
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2)

emo_vecs_tsne = tsne.fit_transform(emo_vecs_np[:,0,:])

import pandas as pd

df = pd.DataFrame(emo_vecs_tsne, columns=['x', 'y'])

df['wav'] = [x[1] for x in emo_vecs]
df['emotion'] = df['wav'].apply(lambda x: x.split('/')[-2])



px.scatter(df, x='x', y='y', hover_data=['wav'], color='emotion')





# Inference using Multi-Speaker

In [495]:


class FusionDecoderV3(nn.Module):
    def __init__(self, hidden_dim, acoustic):
        super(FusionDecoderV3, self).__init__()
        # self.attn = AttentionFusion(prosody_dim, hidden_dim)
        
        self.ff1 = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 128))
        self.ff2 = nn.Sequential(nn.Linear(512+256, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 512))
        self.ff3 = nn.Sequential(nn.Linear(192, 128), nn.ReLU(), nn.Linear(128, 128))

        self.base_model = copy.deepcopy(acoustic)

    def forward(self, units, emo_vecs, spk_vecs, logmels):
        # units: (batch_size, time, hidden_dim)
        # emo_vecs: (batch_size, emo_vec_size)
        # spk_vecs: (batch_size, spk_vec_size)
        # logmels: (batch_size, time, n_mels)
        
        # batch_size, time, _ = units.shape
        
        # Apply attention
        o = self.base_model.encoder(units.cuda())

        o2 = self.ff1(emo_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        o2b = self.ff3(spk_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        # units_attn = self.attn(o, emo_vecs)  # (batch_size, time, hidden_dim)

        o3 = self.ff2(torch.cat([o, o2, o2b], dim=-1))

        d = self.base_model.decoder(o3, logmels)

        return d
    
    def generate(self, units, emo_vecs, spk_vecs):
        # units: (batch_size, time, hidden_dim)
        # emo_vecs: (batch_size, emo_vec_size)
        # spk_vecs: (batch_size, spk_vec_size)
                
        # Apply attention
        o = self.base_model.encoder(units.cuda())

        # units_attn = self.attn(o, emo_vecs)
        o2 = self.ff1(emo_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        o2b = self.ff3(spk_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)

        # units_attn = self.attn(o, emo_vecs)  # (batch_size, time, hidden_dim)

        o3 = self.ff2(torch.cat([o, o2, o2b], dim=-1))


        d = self.base_model.decoder.generate(o3)
        
        return d


In [496]:
acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_discrete", trust_repo=True).cuda()
decoderv3 = FusionDecoderV3(512, acoustic).cuda()


Using cache found in /home/dcor/niskhizov/cache/hub/bshall_acoustic-model_main


In [571]:
decoderv3.load_state_dict(torch.load("decoderV3.pth"))

/tmp/ipykernel_288879/284974761.py:1: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



<All keys matched successfully>

In [572]:
def generate_speechv3(units, emo_vec, spk_vec):
        
        with torch.no_grad():
            spec = decoderv3.generate(units.unsqueeze(0).cuda(), emo_vec.unsqueeze(0).cuda(), spk_vec.cuda())
            target = hifigan(spec.transpose(1, 2)).cpu()[0][0]

        return target, spec

In [574]:
generated_wav_sad2angry, spec = generate_speechv3(embed_sad[0], embed_surprise[1], embed_angry_male[3])
generated_wav_sad2angry_male, spec = generate_speechv3(embed_sad[0], embed_sad[1], embed_angry_male[3])
generated_wav_sad2surprise, spec = generate_speechv3(embed_sad[0], embed_surprise[1], embed_surprise[3])


display(Audio(generated_wav_sad2angry, rate=sr))
display(Audio(generated_wav_sad2angry_male, rate=sr))
display(Audio(generated_wav_sad2surprise, rate=sr))
display(Audio(embed_sad[2], rate=sr))
display(Audio(embed_angry[2], rate=sr))
display(Audio(embed_angry_male[2], rate=sr))
display(Audio(embed_surprise[2], rate=sr))
